# Bank Marketing Campaign Optimization

**Independent research project | Python, SQL, classification, model validation**

Source: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)  
License: CC BY 4.0 | Campaign observations: May 2008-Nov. 2010

## tl;dr

Can response modeling improve pre-call campaign targeting under class imbalance without
using information that is unavailable at the decision point? A calibrated random forest
selected through five-fold training-set validation achieved **0.813 ROC-AUC** and
**0.485 PR-AUC** on an untouched 8,236-row holdout. The top-ranked 20% captured
**65.9% of subscribers** at **3.30x lift**. Bootstrap intervals and a leakage audit
show both uncertainty and why post-call `duration` must be excluded.

## Context & Methods

**Decision:** rank campaign contact records so a fixed call budget reaches more likely
subscribers using only information available before a call begins.

1. Remove exact duplicate rows and preserve `unknown` as an observed category.
2. Reserve a stratified 20% holdout before model selection.
3. Compare a prevalence baseline, class-weighted logistic regression, and random forest
   with five-fold stratified cross-validation on the training set; select by mean PR-AUC.
4. Calibrate the selected model with sigmoid calibration using training data only.
5. Evaluate once on the holdout with ROC-AUC, PR-AUC, calibration, budget lift, capture,
   and 1,000 bootstrap samples for 95% confidence intervals.
6. Refit the same model with post-call `duration` only as a leakage audit.

### Key Assumptions

- Each row is a campaign contact record; the public extract has no customer identifier,
  so repeated clients cannot be grouped during splitting.
- The extract lacks a complete year-level timestamp, preventing a defensible temporal split.
- Ranking identifies likely subscribers, not customers caused to subscribe by a call.
- A production system would require temporal, fairness, consent, contact-frequency,
  calibration-drift, and randomized-incrementality validation.

## Data

In [1]:
from pathlib import Path
import pandas as pd

from analysis import run_analysis

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
PROJECT_DIR = Path.cwd()
PROJECT_DIR

PosixPath('/workspace/scratch/1567abb3224e/github_analytics_portfolio/project_2_bank_campaign_optimization')

In [2]:
# Execute the complete deterministic pipeline and refresh all saved outputs.
results = run_analysis(PROJECT_DIR)
summary = results["summary"]

pd.Series({
    "Raw contact records": summary["raw_rows"],
    "Duplicates removed": summary["duplicate_rows_removed"],
    "Analysis records": summary["analysis_rows"],
    "Positive outcomes": summary["positive_outcomes"],
    "Subscription rate": summary["subscription_rate"],
    "Rows containing literal 'unknown'": summary["rows_with_unknown"],
    "Source SHA-256": summary["data_sha256"],
}, name="Source profile")

Raw contact records                                                              41188
Duplicates removed                                                                  12
Analysis records                                                                 41176
Positive outcomes                                                                 4639
Subscription rate                                                               0.1127
Rows containing literal 'unknown'                                                10698
Source SHA-256                       74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959...
Name: Source profile, dtype: object

In [3]:
# Compact quality checks; 'unknown' is documented rather than silently imputed.
results["data_quality"]

                     Check  Value      Status
0                 Raw rows  41188        Pass
1     Exact duplicate rows     12  Documented
2            Analysis rows  41176        Pass
3            Missing cells      0        Pass
4    Literal unknown cells  12716  Documented
5  Rows containing unknown  10698  Documented
6        Positive outcomes   4639        Pass

## Results

In [4]:
# Candidate selection uses training folds only; the holdout is not consulted.
results["cv_summary"]

                 Model  ROC_AUC_Mean  ROC_AUC_Std  PR_AUC_Mean  PR_AUC_Std
0        Random Forest        0.7997       0.0107       0.4671      0.0182
1  Logistic Regression        0.7906       0.0084       0.4466      0.0147
2       Dummy Baseline        0.5000       0.0000       0.1127      0.0001

In [5]:
# Final performance is measured once on the untouched holdout.
results["holdout_metrics"]

                      Model  ROC_AUC  PR_AUC  Brier_Score  Log_Loss  \
0  Calibrated Random Forest   0.8133  0.4849       0.0754    0.2660   

   Precision_at_0.5  Recall_at_0.5  F1_at_0.5  
0            0.6599         0.2446     0.3569  

In [6]:
# Nonparametric uncertainty around the holdout estimates.
results["bootstrap_intervals"]

          Metric  Estimate  CI95_Lower  CI95_Upper  BootstrapSamples
0        ROC_AUC    0.8133      0.7971      0.8301              1000
1         PR_AUC    0.4849      0.4493      0.5206              1000
2     Top20_Lift    3.2958      3.1561      3.4418              1000
3  Top20_Capture    0.6595      0.6315      0.6887              1000

In [7]:
# Operational performance across call-budget levels.
results["budget_metrics"].assign(
    contact_share=lambda frame: frame["contact_share"].map(lambda value: f"{value:.0%}"),
    conversion_rate=lambda frame: frame["conversion_rate"].map(lambda value: f"{value:.1%}"),
    responder_capture=lambda frame: frame["responder_capture"].map(lambda value: f"{value:.1%}"),
)

  contact_share  contact_records conversion_rate   lift responder_capture
0           10%              824           52.5% 4.6637             46.7%
1           20%             1648           37.1% 3.2958             65.9%
2           30%             2471           27.7% 2.4567             73.7%
3           40%             3295           22.3% 1.9770             79.1%
4           50%             4118           19.1% 1.6940             84.7%
5           60%             4942           16.5% 1.4654             87.9%
6           70%             5766           14.8% 1.3114             91.8%
7           80%             6589           13.3% 1.1772             94.2%
8           90%             7413           12.1% 1.0763             96.9%
9          100%             8236           11.3% 1.0000            100.0%

In [8]:
# A same-architecture audit quantifies the misleading gain from post-call duration.
results["leakage_audit"]

                       Feature_Set  Includes_Duration  ROC_AUC  PR_AUC
0           Pre-call features only              False   0.8139  0.4868
1  Pre-call features plus duration               True   0.9459  0.6523

In [9]:
# SQL-derived descriptive evidence; associations are not interpreted causally.
results["previous_outcome"]

  previous_outcome  contact_records  subscribers  conversion_rate_pct
0          success             1373          894              65.1100
1          failure             4252          605              14.2300
2      nonexistent            35551         3140               8.8300

## Takeaways

- **Robust ranking:** the random forest led training-set model selection with mean
  PR-AUC **0.467 +/- 0.018**; the untouched holdout PR-AUC was **0.485**
  (95% bootstrap CI **0.449-0.521**).
- **Budget relevance:** contacting the top 20% by score captured **65.9%** of holdout
  subscribers (95% CI **63.2%-68.9%**) at **3.30x lift**.
- **Leakage control:** adding `duration`, which is known only after a call, inflated
  same-model holdout ROC-AUC from **0.814 to 0.946** and PR-AUC from **0.487 to 0.652**.
- **Decision boundary:** these results justify a prospective temporal validation and
  randomized pilot; they do not establish causal campaign lift or production readiness.

Rendered figures and machine-readable results are saved under `outputs/`. Run
`python validate_outputs.py` after execution for an independent integrity check.